# PA5 - PySpark Collaborative Data Prep
**Engineer:** Manuel Rafanan  
**Sprint:** 5  
**Dataset:** TeamStatistics.csv

---
### Tasks
1. Sprint 5 Setup
2. TeamStatistics Ingestion
3. Feature Engineering
4. Table 1 - 3PT Efficiency Categories
5. Monthly Trends
6. Save Outputs

---
## Task 1 - Sprint 5 Setup

In [ ]:
# Install PySpark (Colab only - skip if running locally)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "pyspark", "--quiet"])
print("PySpark installed.")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PA5_TeamStatistics")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print(f"PySpark version       : {spark.version}")
print(f"executor.memory       : {spark.conf.get('spark.executor.memory')}")
print(f"shuffle.partitions    : {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"App name              : {spark.sparkContext.appName}")
print(f"Master                : {spark.sparkContext.master}")

---
## Task 2 - Data Prep: TeamStatistics Ingestion

In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

schema = StructType([
    StructField("gameId",                 StringType(),  True),
    StructField("gameDateTimeEst",         StringType(),  True),
    StructField("teamCity",                StringType(),  True),
    StructField("teamName",                StringType(),  True),
    StructField("teamId",                  StringType(),  True),
    StructField("opponentTeamCity",        StringType(),  True),
    StructField("opponentTeamName",        StringType(),  True),
    StructField("opponentTeamId",          StringType(),  True),
    StructField("home",                    IntegerType(), True),
    StructField("win",                     IntegerType(), True),
    StructField("teamScore",               IntegerType(), True),
    StructField("opponentScore",           IntegerType(), True),
    StructField("assists",                 IntegerType(), True),
    StructField("blocks",                  IntegerType(), True),
    StructField("steals",                  IntegerType(), True),
    StructField("fieldGoalsAttempted",     IntegerType(), True),
    StructField("fieldGoalsMade",          IntegerType(), True),
    StructField("fieldGoalsPercentage",    DoubleType(),  True),
    StructField("threePointersAttempted",  IntegerType(), True),
    StructField("threePointersMade",       IntegerType(), True),
    StructField("threePointersPercentage", DoubleType(),  True),
    StructField("freeThrowsAttempted",     IntegerType(), True),
    StructField("freeThrowsMade",          IntegerType(), True),
    StructField("freeThrowsPercentage",    DoubleType(),  True),
    StructField("reboundsDefensive",       IntegerType(), True),
    StructField("reboundsOffensive",       IntegerType(), True),
    StructField("reboundsTotal",           IntegerType(), True),
    StructField("foulsPersonal",           IntegerType(), True),
    StructField("turnovers",               IntegerType(), True),
    StructField("plusMinusPoints",         IntegerType(), True),
    StructField("numMinutes",              IntegerType(), True),
    StructField("q1Points",                IntegerType(), True),
    StructField("q2Points",                IntegerType(), True),
    StructField("q3Points",                IntegerType(), True),
    StructField("q4Points",                IntegerType(), True),
    StructField("benchPoints",             IntegerType(), True),
    StructField("biggestLead",             IntegerType(), True),
    StructField("biggestScoringRun",       IntegerType(), True),
    StructField("leadChanges",             IntegerType(), True),
    StructField("pointsFastBreak",         IntegerType(), True),
    StructField("pointsFromTurnovers",     IntegerType(), True),
    StructField("pointsInThePaint",        IntegerType(), True),
    StructField("pointsSecondChance",      IntegerType(), True),
    StructField("timesTied",               IntegerType(), True),
    StructField("timeoutsRemaining",       IntegerType(), True),
    StructField("seasonWins",              IntegerType(), True),
    StructField("seasonLosses",            IntegerType(), True),
    StructField("coachId",                 StringType(),  True)
])

# TODO: update path for Google Drive if needed
# e.g. CSV_PATH = "/content/drive/MyDrive/PA5/TeamStatistics.csv"
CSV_PATH = "TeamStatistics.csv"

raw_df = (
    spark.read
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .schema(schema)
    .csv(CSV_PATH)
)

print(f"Total rows loaded: {raw_df.count():,}")
raw_df.printSchema()

In [ ]:
REQUIRED_COLS = [
    "gameId",
    "gameDateTimeEst",
    "teamName",
    "teamScore",
    "assists",
    "turnovers",
    "threePointersMade",
    "threePointersAttempted",
    "win"
]

team_df = raw_df.select(REQUIRED_COLS)

print(f"Rows after column selection: {team_df.count():,}")
team_df.show(5, truncate=False)

---
## Task 3 - Transforms: Feature Engineering

In [ ]:
from pyspark.sql import functions as F

# Parse date string to timestamp
team_df = team_df.withColumn(
    "game_ts",
    F.to_timestamp(F.col("gameDateTimeEst"), "yyyy-MM-dd HH:mm:ss")
)

# year_month: YYYY-MM format
team_df = team_df.withColumn(
    "year_month",
    F.date_format(F.col("game_ts"), "yyyy-MM")
)

# season_year: 4-digit year
team_df = team_df.withColumn(
    "season_year",
    F.year(F.col("game_ts"))
)

# three_pt_pct: null when attempts = 0 to avoid divide-by-zero
team_df = team_df.withColumn(
    "three_pt_pct",
    F.when(
        F.col("threePointersAttempted") == 0, None
    ).otherwise(
        F.round(F.col("threePointersMade") / F.col("threePointersAttempted"), 4)
    )
)

# efficiency: (teamScore + assists) / turnovers, null when turnovers = 0
team_df = team_df.withColumn(
    "efficiency",
    F.when(
        F.col("turnovers") == 0, None
    ).otherwise(
        F.round((F.col("teamScore") + F.col("assists")) / F.col("turnovers"), 4)
    )
)

print("Feature engineering complete. Sample rows:")
team_df.select(
    "gameId", "teamName", "year_month", "season_year", "three_pt_pct", "efficiency"
).show(10, truncate=False)

In [ ]:
# Sanity check: verify divide-by-zero protection
bad_3pt = team_df.filter(
    (F.col("threePointersAttempted") == 0) & F.col("three_pt_pct").isNotNull()
).count()

bad_eff = team_df.filter(
    (F.col("turnovers") == 0) & F.col("efficiency").isNotNull()
).count()

print(f"Rows where 3PA=0 but three_pt_pct not null (should be 0): {bad_3pt}")
print(f"Rows where turnovers=0 but efficiency not null (should be 0): {bad_eff}")

---
## Task 4 - Output: Table 1 - 3PT Efficiency Categories

In [ ]:
team_with_cat = team_df.withColumn(
    "three_pt_category",
    F.when(F.col("three_pt_pct").isNull(),  "No Attempts")
     .when(F.col("three_pt_pct") >= 0.40,   "Elite (>=40%)")
     .when(F.col("three_pt_pct") >= 0.35,   "Above Average (35-39%)")
     .when(F.col("three_pt_pct") >= 0.30,   "Average (30-34%)")
     .otherwise(                             "Below Average (<30%)")
)

table1 = (
    team_with_cat
    .groupBy("three_pt_category")
    .agg(
        F.count("*").alias("game_count"),
        F.round(F.avg("three_pt_pct"), 4).alias("avg_three_pt_pct"),
        F.round(F.avg("teamScore"), 2).alias("avg_teamScore"),
        F.round(F.avg("efficiency"), 4).alias("avg_efficiency"),
        F.round(F.avg("win"), 4).alias("win_rate")
    )
    .orderBy(F.desc("avg_three_pt_pct"))
)

print("=== Table 1: 3PT Efficiency Categories ===")
table1.show(truncate=False)

---
## Task 5 - Output: Monthly Trends

In [ ]:
monthly_trends = (
    team_df
    .filter(F.col("year_month").isNotNull())
    .groupBy("year_month")
    .agg(
        F.count("*").alias("games_in_month"),
        F.round(F.avg("teamScore"), 2).alias("avg_teamScore"),
        F.round(F.avg("assists"), 2).alias("avg_assists"),
        F.round(F.avg("efficiency"), 4).alias("avg_efficiency")
    )
    .orderBy("year_month")
)

print("=== Monthly Trends ===")
monthly_trends.show(50, truncate=False)

---
## Task 6 - Engineering: Save Outputs

In [ ]:
import os

# Update if using Google Drive:
# BASE_OUTPUT_DIR = "/content/drive/MyDrive/PA5/outputs"
BASE_OUTPUT_DIR = "outputs/pa5"

PATHS = {
    "engineered": f"{BASE_OUTPUT_DIR}/team_engineered",
    "table1":     f"{BASE_OUTPUT_DIR}/table1_3pt_categories",
    "monthly":    f"{BASE_OUTPUT_DIR}/monthly_trends"
}

def save_csv(df, path, label):
    (
        df.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(path)
    )
    print(f"Saved {label} -> {path}")

save_csv(team_df,        PATHS["engineered"], "Engineered team data")
save_csv(table1,         PATHS["table1"],     "Table 1 - 3PT categories")
save_csv(monthly_trends, PATHS["monthly"],    "Monthly trends")

print("\nAll outputs saved.")

In [ ]:
for label, path in PATHS.items():
    if os.path.exists(path):
        files = [f for f in os.listdir(path) if f.endswith(".csv")]
        print(f"{label:12s} -> {path}  [{len(files)} CSV file(s)]")
    else:
        print(f"{label:12s} -> {path}  [NOT FOUND]")